# Fase 4 — Notebook integrador: análisis, reproducibilidad y comunicación de resultados

**Proyecto:** Grupo 3 · MCDI500 · **Dataset:** Siniestros de tipo atropello 2023 (CONASET / Carabineros de Chile)

**Pregunta del proyecto:** ¿qué factores de la vía, del entorno y del contexto del siniestro se asocian con que un atropello resulte con al menos un fallecido?

Este notebook integra las cuatro fases y se ejecuta de principio a fin con *Restart Kernel and Run All Cells*:

| Fase | Qué aporta | Dónde está |
|---|---|---|
| F1 | Problema, entorno y reconocimiento del dato | `F1/F1_Definicion.ipynb` |
| F2 | Obtención, limpieza y variables nuevas | `F2/src/` |
| F3 | Corrección de la hora y mediciones de eficiencia | `F3/src/` |
| F4 | Clases de medición, pruebas, visualizaciones | `F4/src/`, `F4/tests/` |

Se ejecuta desde la carpeta `F4/`. El CSV crudo no está versionado: ver el README para obtenerlo y verificar su huella.

## 1. Configuración y verificación del entorno

In [ ]:
import sys
from pathlib import Path

import matplotlib
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn

for ruta in ("../F2/src", "../F3/src", "src"):          # módulos de cada fase
    sys.path.insert(0, str(Path(ruta).resolve()))

import mediciones
import pipeline
import visualizacion

visualizacion.configurar_tema()
print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__, "| NumPy:", np.__version__, "| scikit-learn:", sklearn.__version__)
print("Matplotlib:", matplotlib.__version__, "| seaborn:", sns.__version__)
print("Entorno virtual activo:", sys.prefix != sys.base_prefix)

Path("resultados").mkdir(exist_ok=True)           # evidencia versionada del entorno y de las mediciones
Path("figuras").mkdir(exist_ok=True)
Path("resultados/entorno.txt").write_text(
    f"python={sys.version.split()[0]}\npandas={pd.__version__}\nnumpy={np.__version__}\n"
    f"scikit-learn={sklearn.__version__}\nmatplotlib={matplotlib.__version__}\nseaborn={sns.__version__}\n"
    f"venv_activo={sys.prefix != sys.base_prefix}\n", encoding="utf-8")

In [ ]:
RUTA_CRUDO = Path("../F1/data/raw/Atropellos_2023.csv")
huella = pipeline.huella_sha256(RUTA_CRUDO)
print("SHA-256 del dato crudo:", huella)
print("Coincide con el dato de origen documentado:", huella == pipeline.HUELLA_SHA256)

## 2. Fases 1 a 3 integradas: del dato crudo al conjunto preparado

`pipeline.preparar_dataset()` encadena las funciones ya existentes: `obtencion.cargar_crudo` → `verificar_esquema` → `limpieza.limpiar_datos` → `transformacion.generar_nuevas_variables` → `pipeline.corregir_hora`.

**Corrección heredada de la Fase 3.** `Hora_limpia` quedó en 0 en las 5.541 filas de la Fase 2 porque se calculó desde `Hora`, que en el archivo original vale lo mismo en todas las filas. La hora real está en `Hora_texto`. Se corrige acá, sin modificar F2, y la prueba de regresión comprueba que ninguna otra columna cambia.

In [ ]:
df = pipeline.preparar_dataset(RUTA_CRUDO)

assert df.shape[0] == 5541, "El pipeline perdió o duplicó filas"
assert int(df["Es_Fatal"].sum()) == 407, "Cambió la cantidad de casos fatales"
print("Filas:", df.shape[0], "· columnas:", df.shape[1])
print(f"Casos fatales: {int(df['Es_Fatal'].sum())} ({df['Es_Fatal'].mean():.2%})")

blancos = {c: int((df[c].astype(str).str.strip() == "").sum()) for c in pipeline.COLUMNAS_TEXTO}
print("Celdas en blanco tras la limpieza:", blancos)
assert sum(blancos.values()) == 0

In [ ]:
import limpieza, obtencion, transformacion

previo = transformacion.generar_nuevas_variables(limpieza.limpiar_datos(obtencion.cargar_crudo(RUTA_CRUDO)))
print("Hora_limpia en la Fase 2 -> valores distintos:", previo["Hora_limpia"].nunique(), "| valores:", previo["Hora_limpia"].unique())
print("Hora_limpia corregida   -> valores distintos:", df["Hora_limpia"].nunique(), "| rango:", df["Hora_limpia"].min(), "a", df["Hora_limpia"].max())
assert df["Hora_limpia"].between(0, 23).all()

## 3. Preprocesamiento: dos conjuntos con propósitos distintos

El conjunto que sirve para modelar no sirve para graficar: con *one hot* las categorías pasan a columnas de ceros y unos y con el escalamiento los valores pierden su unidad. Por eso se conservan dos versiones:

- **`visual`**: copia interpretable, con categorías y unidades originales (hora en horas, zona y causa con su nombre).
- **Matriz de análisis**: partición estratificada 80/20 con `random_state=42`; el escalador y el codificador se **ajustan solo con entrenamiento** y se aplican a prueba, para evitar fuga de datos.

In [ ]:
visual = pipeline.construir_df_visual(df)
A_tr, A_te, y_tr, y_te, nombres = pipeline.construir_matriz_analisis(df)

print("Para visualizar:", visual.shape, "| columnas:", list(visual.columns))
print("Para análisis  : entrenamiento", A_tr.shape, "| prueba", A_te.shape, "| columnas:", len(nombres))
assert not (np.isnan(A_tr).any() or np.isnan(A_te).any()), "La matriz tiene nulos"
assert A_tr.shape[1] == A_te.shape[1] == len(nombres)
print(f"Proporción de fatales: entrenamiento {y_tr.mean():.2%} · prueba {y_te.mean():.2%}")
visual.head()

## 4. Eficiencia: tres comparaciones con clases `Medicion` (patrón Strategy)

`Medicion` es la clase base; cada hija declara solo su entrada y sus dos estrategias (`version_a`, `version_b`). Cada comparación verifica primero que A y B entreguen el mismo resultado y luego mide **tiempo** (mínimo de 7 repeticiones, `timeit`) y **memoria pico** (`tracemalloc`) sobre **tamaños crecientes**. El exponente *k* de la relación t ≈ c·nᵏ se estima con la pendiente log-log (`numpy.polyfit`).

In [ ]:
N = len(df)
corridas = [
    mediciones.MedicionHora([500, 1000, 2500, N]),
    mediciones.MedicionBusqueda([500, 1000, 2500, N]),
    mediciones.MedicionRecorrido([1000, 5000, 20000]),
]
tablas = []
for medicion in corridas:                    # polimorfismo: la misma llamada para las tres
    tabla = medicion.ejecutar(df)
    tablas.append(tabla)
    k = medicion.exponentes()
    print(f"{medicion.nombre}: A = {medicion.etiqueta_a} | B = {medicion.etiqueta_b}")
    print(f"  exponente empírico  k_A = {k['k_a']:.2f}   k_B = {k['k_b']:.2f}")
resumen = pd.concat(tablas, ignore_index=True)
resumen.to_csv("resultados/mediciones.csv", index=False)
resumen.round(5)

In [ ]:
for tabla, medicion in zip(tablas, corridas):
    ultima = tabla.iloc[-1]
    print(f"[{medicion.nombre}] con n = {int(ultima['n']):,}: "
          f"B es {ultima['razon_tiempo']:.1f} veces más rápida; "
          f"memoria A = {ultima['memoria_a_mb']:.3f} MB, B = {ultima['memoria_b_mb']:.3f} MB "
          f"(razón A/B = {ultima['razon_memoria']:.2f}).")
    invertidos = tabla.loc[tabla["razon_tiempo"] < 1, "n"].tolist()
    if invertidos:
        print(f"   En n = {invertidos} la versión A resultó más rápida (costo fijo de B).")
    else:
        print("   B fue más rápida en todos los tamaños medidos.")

## 5. Visualizaciones analíticas

Hilo narrativo: **contexto** (cuántos atropellos hay y a qué hora se vuelven letales), **contraste** (qué causas concentran la letalidad) y **resolución** (si el efecto horario se mantiene al separar por zona). Se usan **tasas con su n** y no cifras absolutas, porque los grupos son de tamaños muy distintos y el 7,35 % de casos fatales hace que los conteos engañen.

In [ ]:
fig1 = visualizacion.figura_hora(visual)
fig1.savefig("figuras/fig1_hora.png", dpi=170, bbox_inches="tight")

In [ ]:
por_hora = visual.groupby("Hora_limpia")["Es_Fatal"].agg(n="size", tasa="mean")
pico = por_hora["n"].idxmax()
mad = visual.loc[visual["Hora_limpia"].between(0, 5), "Es_Fatal"]
dia = visual.loc[visual["Hora_limpia"].between(6, 17), "Es_Fatal"]
print(f"Qué muestra : la mayor cantidad de atropellos ocurre a las {pico}:00 h ({int(por_hora.loc[pico, 'n'])} casos), "
      f"pero la letalidad es de {mad.mean():.1%} entre 0 y 5 h (n={len(mad)}) y de {dia.mean():.1%} entre 6 y 17 h (n={len(dia)}).")
print("Qué infiero : el momento con más atropellos no es el de mayor riesgo de muerte; la madrugada concentra pocos casos pero muy graves.")
print("Límite      : no hay datos de flujo de peatones ni vehículos por hora, así que no se puede hablar de riesgo por exposición.")

In [ ]:
fig2 = visualizacion.figura_causa(visual)
fig2.savefig("figuras/fig2_causa.png", dpi=170, bbox_inches="tight")

In [ ]:
tc = pipeline.tasa_fatalidad(visual, "Causa")
print(tc.round(1).to_string(index=False))
print("\nQué muestra : la letalidad varía mucho según la causa asignada; las dos mayores superan el doble del promedio general.")
print("Qué infiero : la gravedad se asocia más a quién y a qué condición originó el atropello que a su frecuencia.")
print("Límite      : la causa la asigna Carabineros (atribución, no medición) y se excluyen las causas con n < 30.")

In [ ]:
fig3 = visualizacion.figura_zona_tramo(visual)
fig3.savefig("figuras/fig3_zona_tramo.png", dpi=170, bbox_inches="tight")

In [ ]:
noche = visual["Hora_limpia"].isin([22, 23, 0, 1, 2, 3, 4, 5])
estratos = visual.assign(noche=noche).groupby(["Zona", "noche"])["Es_Fatal"].agg(n="size", fatales="sum", tasa="mean")
estratos["tasa"] = (estratos["tasa"] * 100).round(1)
print(estratos.to_string())
zona = visual.groupby("Zona")["Es_Fatal"].agg(n="size", fatales="sum", tasa="mean")
zona["tasa"] = (zona["tasa"] * 100).round(1)
print("\nTasa de fatalidad por zona:")
print(zona.to_string())
print("\nQué muestra : dentro de cada zona, la franja 22-05 h tiene más letalidad que el resto del día.")
print("Qué infiero : el efecto nocturno no se explica solo por ser zona rural; se mantiene al estratificar.")
print("Límite      : es una asociación observacional; no se controlan velocidad, iluminación ni consumo de alcohol, y no establece causalidad.")

## 6. Validación técnica: pruebas de casos normales, límite y excepciones

Las pruebas viven en `F4/tests/test_pipeline.py` y se ejecutan acá mismo, de modo que el resultado queda como evidencia en el notebook ejecutado y en su exportación a HTML.

In [ ]:
import subprocess

resultado = subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"],
                           capture_output=True, text=True)
print(resultado.stderr)
assert resultado.returncode == 0, "Alguna prueba falló: revisar el detalle de arriba"

## 7. Conclusiones y limitaciones

- **Hallazgos.** La letalidad de los atropellos depende de cuándo y dónde ocurren: se concentra en la madrugada y en zona rural, y el efecto nocturno se mantiene dentro de cada zona. Las causas atribuidas al alcohol del conductor y a la imprudencia del peatón presentan las tasas más altas.
- **Eficiencia.** En las tres comparaciones se verificó primero la equivalencia de resultados; las razones de tiempo y memoria se leen en la tabla de la sección 4, que se genera en cada ejecución.
- **Limitaciones.** Un solo año (2023); posible subregistro de fallecidos posteriores a 48 h; causa asignada por Carabineros; sin datos de exposición; asociaciones no causales.
- **Mejoras propuestas.** Incorporar más años para estabilizar las tasas de los grupos pequeños; modelar la probabilidad de fatalidad con regresión logística usando la matriz de análisis ya preparada; incluir variables de exposición (flujo vehicular y peatonal).